# Ttoken Studio - GPU de video
1. Ambiente de execucao > GPU T4  2. Segredos (chave) > COLAB_SECRET = senha do SnapDeploy (so uma vez)  3. Rode a celula e deixe rodando ate aparecer *registrado no backend*.

In [ ]:
# TTOKEN STUDIO - Colab GPU de video (1 celula so). Rode e deixe rodando.
import os, sys, subprocess, getpass, urllib.request

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers",
                "accelerate", "sentencepiece", "imageio", "imageio-ffmpeg"], check=True)
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "cloudflared"); os.chmod("cloudflared", 0o755)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/fabiopimentapinto16-ai/ttokem-backend/main/colab/server.py",
    "server.py")

print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
      capture_output=True, text=True).stdout.strip() or "NENHUMA - troque para GPU T4")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive nao montado:", e)

# Senha: fica guardada nos "Segredos" do Colab (icone de chave, na esquerda), nome COLAB_SECRET.
segredo = ""
try:
    from google.colab import userdata
    segredo = userdata.get("COLAB_SECRET") or ""
except Exception:
    pass
if not segredo:
    segredo = getpass.getpass("COLAB_SECRET (mesma do SnapDeploy): ")

os.environ["BACKEND_URL"] = "https://ttokem-backend-8ba2a.containers.snapdeploy.app"
os.environ["COLAB_SECRET"] = segredo
os.environ["COLAB_MODEL"] = "cogvideox-2b"

s = open("server.py").read()
alvo = 'OUT = tempfile.mkdtemp(prefix="ttoken_")'
if alvo in s and os.path.isdir("/content/drive/My Drive"):
    open("server.py", "w").write(s.replace(alvo,
        'OUT = "/content/drive/My Drive/SnapDeploy_Videos"\nos.makedirs(OUT, exist_ok=True)', 1))

p = subprocess.Popen([sys.executable, "-u", "server.py"], stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
for linha in p.stdout:
    print(linha, end="")
